# Camera Installation Decision for Vending Machines

本 Notebook 以歷史 3,000 台販賣機資料建立分類模型，並對 1,000 個 candidate locations 產出 camera 部署建議。
主要輸出會寫入 `outputs/figures/`、`outputs/tables/` 與 `outputs/report.md`。

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display
import pandas as pd

from classification_case_analysis import (
    BUSINESS_THRESHOLD,
    ROOT,
    build_descriptive_tables,
    build_history_frames,
    build_report,
    budget_scenarios,
    choose_selected_model,
    fit_models,
    plot_business_figures,
    plot_descriptive_figures,
    plot_model_figures,
    run_sensitivity_analysis,
    save_table,
    score_candidates,
    summarize_high_risk,
    AnalysisArtifacts,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
ROOT

## 1. 載入資料與欄位 mapping
先把歷史資料與 candidate 資料都整理成 location-level，再確認資料品質。

In [ ]:
history_location, history_daily, candidate_location, mapping_table, data_quality = build_history_frames()
mapping_table

In [ ]:
data_quality

## 2. Descriptive analysis
依 Broken status 產出 summary statistics 與視覺化。

In [ ]:
descriptive_tables = build_descriptive_tables(history_location)
descriptive_figures = plot_descriptive_figures(history_location, history_daily)
for name, table in descriptive_tables.items():
    save_table(table, f'{name}.csv')
descriptive_tables['overview']

In [ ]:
descriptive_tables['binary_summary']

In [ ]:
display(Image(filename=str(descriptive_figures['broken_distribution'])))
display(Image(filename=str(descriptive_figures['building_type_broken_rate'])))
display(Image(filename=str(descriptive_figures['distance_boxplot'])))

## 3. Classification modeling
建立 Logistic Regression 與 Random Forest，並在 test set 比較 threshold 0.5 與 0.2 的表現。

In [ ]:
model_results, model_comparison, split_data = fit_models(history_location)
X_train, X_test, y_train, y_test = split_data
model_figures = plot_model_figures(model_results, X_test, y_test)
save_table(model_comparison, 'model_comparison.csv')
save_table(model_results['Logistic Regression'].feature_table, 'logistic_coefficients.csv')
save_table(model_results['Random Forest'].feature_table, 'random_forest_feature_importance.csv')
model_comparison

In [ ]:
model_results['Logistic Regression'].feature_table.head(10)

In [ ]:
model_results['Random Forest'].feature_table.head(10)

In [ ]:
display(Image(filename=str(model_figures['roc_curve'])))

## 4. Business recommendation
用商業門檻 $p > 0.2$ 對 candidate locations 做部署決策，並比較 outdoor-only human rule。

In [ ]:
selected_model_name = choose_selected_model(model_comparison)
all_scores, selected_scores, strategy_comparison, risk_profile = score_candidates(
    candidate_location,
    model_results,
    selected_model_name,
)
selected_scores.to_csv(ROOT / 'outputs' / 'tables' / 'camera_recommendations.csv', index=False)
save_table(strategy_comparison, 'strategy_comparison.csv')
save_table(risk_profile, 'risk_profile.csv')
selected_model_name, selected_scores['install_camera'].sum()

In [ ]:
strategy_comparison

In [ ]:
high_risk_summary = summarize_high_risk(selected_scores)
sensitivity_table = run_sensitivity_analysis(selected_scores)
budget_table = budget_scenarios(selected_scores)
save_table(high_risk_summary, 'high_risk_summary.csv')
save_table(sensitivity_table, 'sensitivity_analysis.csv')
save_table(budget_table, 'budget_scenarios.csv')
business_figures = plot_business_figures(selected_scores, sensitivity_table)
high_risk_summary

In [ ]:
sensitivity_table

In [ ]:
display(Image(filename=str(business_figures['sensitivity_total_cost'])))
display(Image(filename=str(business_figures['sensitivity_camera_count'])))
display(Image(filename=str(business_figures['sensitivity_threshold'])))

## 5. 匯出 Markdown 報告
將所有關鍵表格、圖表與解讀整理成 `outputs/report.md`。

In [ ]:
artifacts = AnalysisArtifacts(
    history_location=history_location,
    history_daily=history_daily,
    candidate_location=candidate_location,
    mapping_table=mapping_table,
    data_quality=data_quality,
    descriptive_tables=descriptive_tables,
    model_results=model_results,
    model_comparison=model_comparison,
    selected_model_name=selected_model_name,
    selected_candidate_scores=selected_scores,
    strategy_comparison=strategy_comparison,
    sensitivity_table=sensitivity_table,
    budget_table=budget_table,
    report_path=ROOT / 'outputs' / 'report.md',
)
all_figures = {}
all_figures.update(descriptive_figures)
all_figures.update(model_figures)
all_figures.update(business_figures)
report_path = build_report(artifacts, all_figures)
report_path

In [ ]:
display(Markdown((ROOT / 'outputs' / 'report.md').read_text(encoding='utf-8')[:4000]))